In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df = pd.read_csv("../part3_synthetic_data/data/sales.csv", parse_dates=["date"])

print("Размер таблицы:", df.shape)
print("Период:", df["date"].min().date(), "-", df["date"].max().date())
print("Категории:", list(df["category"].unique()))
df.head()

In [ ]:

limit = df.groupby("product")["qty"].transform(lambda s: s.quantile(0.97))

print("Максимум qty до обработки:", df["qty"].max())
df["qty"] = df["qty"].clip(upper=limit)
print("Максимум qty после обработки:", df["qty"].max())

In [ ]:

weekly = df.groupby(["category", pd.Grouper(key="date", freq="W")])["qty"].sum().unstack(level=0)


weekly = weekly.iloc[1:-1]

print("Количество недель:", len(weekly))
weekly.head()

In [ ]:
weekly.plot(figsize=(11, 5))
plt.title("Недельные продажи по категориям")
plt.xlabel("Неделя")
plt.ylabel("Продано, шт.")
plt.grid(True)
plt.show()

In [ ]:
test_weeks = 8

train = weekly.iloc[:-test_weeks]
test = weekly.iloc[-test_weeks:]

print("Обучающая часть:", len(train), "недель:", train.index[0].date(), "-", train.index[-1].date())
print("Тестовая часть: ", len(test), "недель:", test.index[0].date(), "-", test.index[-1].date())

In [ ]:
N = 4   


forecast_ma = pd.DataFrame(index=test.index, columns=test.columns, dtype=float)
forecast_trend = pd.DataFrame(index=test.index, columns=test.columns, dtype=float)


x_train = np.arange(len(train))
x_test = np.arange(len(train), len(weekly))

for cat in weekly.columns:

    mean_last = train[cat].tail(N).mean()
    forecast_ma[cat] = mean_last


    coeffs = np.polyfit(x_train, train[cat].values, deg=1)  
    forecast_trend[cat] = np.polyval(coeffs, x_test)         

forecast_ma.round(0)

In [ ]:

results = []

for cat in weekly.columns:
    mae_ma = np.mean(np.abs(test[cat] - forecast_ma[cat]))
    mae_trend = np.mean(np.abs(test[cat] - forecast_trend[cat]))
    results.append([cat, round(mae_ma, 1), round(mae_trend, 1)])

mae_table = pd.DataFrame(results, columns=["Категория", "MAE скользящее среднее", "MAE линейный тренд"])
mae_table

In [ ]:
categories = list(weekly.columns)

fig, axes = plt.subplots(len(categories), 1, figsize=(11, 4 * len(categories)))

for i in range(len(categories)):
    cat = categories[i]
    ax = axes[i]


    ax.plot(train.index[-20:], train[cat].tail(20), color="gray", label="обучающие данные")

    ax.plot(test.index, test[cat], color="black", marker="o", label="факт (тест)")

    ax.plot(test.index, forecast_ma[cat], color="blue", linestyle="--", marker="s", label="скользящее среднее (N=4)")
    ax.plot(test.index, forecast_trend[cat], color="red", linestyle="--", marker="^", label="линейный тренд")

    ax.set_title("Категория: " + cat)
    ax.set_xlabel("Неделя")
    ax.set_ylabel("Продано, шт.")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()